# REGIME DEFINING MODEL SIMULATOR

This is a simluator for the regime defining model.

When a regime shift occurs the code creates a new plot

# Simulation

In [ ]:
import numpy as np # for computation of math functions 
from scipy.stats import genextreme # random variable continuous used for getting extreme events
import matplotlib.pyplot as plt# temporary
import plotly.graph_objects as go
import time







fig = go.FigureWidget() # instaciating it the figure widget 
fig2 = go.FigureWidget()
scatter1 = fig.add_scatter(name='Contract price')
scatter2 = fig.add_scatter(name='Lower bound')
scatter3 = fig.add_scatter(name='Upper bound')
scatter4 = fig2.add_scatter(name='Contract price')
scatter5 = fig2.add_scatter(name='Lower bound')
scatter6 = fig2.add_scatter(name='Upper bound')

    
    
class Simulator:
    def __init__(self,extreme_event_probability_historical:float,extreme_event_probability_forward:float,lamb:float,historical_drift_scalar:float,forward_drift_scalar:float,seed:int,regime_lookBack:int) -> None:
        self.extreme_event_probability_historical = float(extreme_event_probability_historical)
        self.extreme_event_probability_forward = float(extreme_event_probability_forward)
        
        
        self.lamb = lamb
        self.historical_drift_scalar = historical_drift_scalar
        self.forward_drift_scalar = forward_drift_scalar
        self.seed = seed
        self.regime_lookBack = regime_lookBack
        np.random.seed(self.seed)
        self.regime_shift = False
        
    def run_historical(self,steps):
        
        pt = np.zeros(steps)
        random_start_value = np.random.randint(1,99)
        
        pt[0] = random_start_value
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_historical,steps)
        
        drift_term = np.random.normal(0,self.historical_drift_scalar,steps)
       
        # have the figure here and have the function here
        for i in range(1,steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
        
        self.historical_log = np.log(pt)
        self.historical_price = pt[-1]
        self.historical_mean = pt.mean()
        
        
        self.lt = (1 - self.historical_log.var() * self.lamb) * (self.historical_log[-1] - self.historical_log.std())
        
        self.ut = (1 + self.historical_log.var() * self.lamb) * (self.historical_log[-1] + self.historical_log.std())
        
        self.lt = np.full(steps,self.lt)
        self.ut = np.full(steps,self.ut)
        plt.plot(pt)
        plt.title(f'Historical_data, [seed: {self.seed}]')
        plt.xlabel('steps')
        plt.ylabel('Price in cents')
        plt.show()
        
       
    
    
    
    
    
    
    
    
    def simulate_forward(self,steps:int): # needs to return most recent price 
        
        
        pt = np.full(steps,self.historical_price)
        
        
        pt[0] = self.historical_price
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_forward,steps)
        
        drift_term = np.random.normal(0,self.forward_drift_scalar,steps)
        
        
        
        
        display(fig)
        # have the figure here and have the function here
        for i in range(1,steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] *  self.historical_mean/steps + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
            pt[i:] = np.full(steps-i,pt[i])
            
            
            if not self.regime_shift and np.log(pt[i]) > self.ut[i]:
                
                print(f'Regime_shift at step {i}: UPPER')
                
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                fig2.data[0].y = pt
                fig2.data[1].y = np.exp(self.lt_regime)
                fig2.data[2].y = np.exp(self.ut_regime)
                fig.data[0].y = pt
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(0.4)
            
            
            
            elif not self.regime_shift and np.log(pt[i]) < self.lt[i]:
                
                print(f'Regime_shift at step {i}: LOWER')
                
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                fig2.data[0].y = pt
                fig2.data[1].y = np.exp(self.lt_regime)
                fig2.data[2].y = np.exp(self.ut_regime)
                fig.data[0].y = pt
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(0.4)
                
                
            
            
            else:
                if self.regime_shift:
                    if np.log(pt[i]) < self.lt_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        fig2.data[0].y = pt
                        fig2.data[1].y = np.exp(self.lt_regime)
                        fig2.data[2].y = np.exp(self.ut_regime)
                        fig.data[0].y = pt
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(0.4)
                        
                    elif np.log(pt[i]) > self.ut_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                        
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        fig2.data[0].y = pt
                        fig2.data[1].y = np.exp(self.lt_regime)
                        fig2.data[2].y = np.exp(self.ut_regime)
                        fig.data[0].y = pt
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(0.4)
                    
                    else:
                        fig2.data[0].y = pt
                        fig2.data[1].y = np.exp(self.lt_regime)
                        fig2.data[2].y = np.exp(self.ut_regime)
                        fig.data[0].y = pt
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        
                        
                        time.sleep(0.4)
                    
                else:
                    fig.data[0].y = pt
                    fig.data[1].y = np.exp(self.lt)
                    fig.data[2].y = np.exp(self.ut)
                    time.sleep(0.4)
                
            
        
        

sim = Simulator(extreme_event_probability_historical=0.005,
              extreme_event_probability_forward=0.005,
              lamb=0.008,
              historical_drift_scalar=0.3,
              forward_drift_scalar=1,
              seed=1123,
              regime_lookBack=2)
sim.run_historical(100)
sim.simulate_forward(100)



# Plot simulation with historical

In [ ]:
import numpy as np # for computation of math functions 
from scipy.stats import genextreme # random variable continuous used for getting extreme events
import matplotlib.pyplot as plt# temporary
import plotly.graph_objects as go
import time
import scipy.stats as stats






fig = go.FigureWidget() # instaciating it the figure widget 
fig.update_layout(title=dict(text="Price path with historical data with static boundary"),
                  xaxis=dict(title=dict(text="Time"))
                  ,yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))
fig2 = go.FigureWidget()
fig2.update_layout(title=dict(text="Forward price path with dynamic boundary"), 
                  xaxis=dict(title=dict(text="Time")),
                  yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))



fig3 = go.FigureWidget()
fig3.update_layout(title=dict(text="Price path with historical data with dynamic boundary"), 
                   xaxis=dict(title=dict(text="Time")),
                   yaxis=dict(title=dict(text="Cent price")),
                   font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))

fig4 = go.FigureWidget()
fig4.update_layout(title=dict(text="Wallet wealth path"), 
                   xaxis=dict(title=dict(text="Time")),
                   yaxis=dict(title=dict(text="Dollars in Wallet")),
                   font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))

scatter1 = fig.add_scatter(name='Contract price')
scatter2 = fig.add_scatter(name='Lower bound')
scatter3 = fig.add_scatter(name='Upper bound')


scatter4 = fig2.add_scatter(name='Contract price')
scatter5 = fig2.add_scatter(name='Lower bound')
scatter6 = fig2.add_scatter(name='Upper bound')

scatter7 = fig3.add_scatter(name='Contract price')
scatter8 = fig3.add_scatter(name='Lower bound')
scatter9 = fig3.add_scatter(name='Upper bound')

scatter10 = fig4.add_scatter(name='Wallet wealth')


class OrderBook:
    
        
    def get_wallet_development(self,st:int,spread:int,wallet:int,fill_probability:float) -> int: 
        
        
        fill_or_not = np.random.binomial(1,fill_probability,1)
        
        if fill_or_not:
            bid = self.get_bid_price(st)
            ask = self.get_ask_price(st,spread)
            bid_ask_spread = ask - bid
        
            wallet += bid_ask_spread
            
            return wallet
        else:
            
            return wallet
        
    def get_bid_price(self,st) -> int:
        
        order_mean_price = st
        
        orderBook = np.full(1 ,order_mean_price) # vector filled with the amount of orders 

        "iterate over the amount of desired incoming orders"
        for i in range(100):
            order_amount = np.random.randint(1,99)
            IncomingPrices = np.round(np.random.standard_gamma(order_mean_price,order_amount),decimals=0) # new random data
            orderBook = np.append(orderBook,IncomingPrices) # adding the new random data to the order book
    
        kernel = stats.gaussian_kde(orderBook)
        x_grid = np.linspace(orderBook.min(),orderBook.max(),1000)
        kde_values = kernel(x_grid)
        price_at_max = x_grid[np.argmax(kde_values)]
            
        
        return price_at_max
        
        
        
        
        
        
    def get_ask_price(self,st:int,spread:int) -> int:
        
        order_mean_price = st + spread
        
        orderBook = np.full(1 ,order_mean_price) # vector filled with the amount of orders 


        "iterate over the amount of desired incoming orders"
        for i in range(100):
            order_amount = np.random.randint(1,99)
            IncomingPrices = np.round(np.random.standard_gamma(order_mean_price,order_amount),decimals=0) # new random data
            orderBook = np.append(orderBook,IncomingPrices) # adding the new random data to the order book
    
        kernel = stats.gaussian_kde(orderBook)
        x_grid = np.linspace(orderBook.min(),orderBook.max(),1000)
        kde_values = kernel(x_grid)
        price_at_max = x_grid[np.argmax(kde_values)]
            
        
        return price_at_max
        
    
    
class Simulator:
    def __init__(self,extreme_event_probability_historical:float,extreme_event_probability_forward:float,lamb:float,historical_drift_scalar:float,forward_drift_scalar:float,seed:int,regime_lookBack:int,historical_steps:int,forward_steps:int,tick_time_seconds:float,wallet_value:int,fill_probability:float,with_wallet: bool) -> None:
        
        self.extreme_event_probability_historical = float(extreme_event_probability_historical)
        self.extreme_event_probability_forward = float(extreme_event_probability_forward)
        self.wallet = np.full(forward_steps+historical_steps,wallet_value)
        self.orderbook = OrderBook()
        self.lamb = lamb
        self.historical_drift_scalar = historical_drift_scalar
        self.forward_drift_scalar = forward_drift_scalar
        self.seed = seed
        self.regime_lookBack = regime_lookBack
        np.random.seed(self.seed)
        self.regime_shift = False
        self.historical_steps = historical_steps
        self.forward_steps = forward_steps
        self.tick_time_seconds = tick_time_seconds
        
        self.spread = np.random.gamma(2,2,forward_steps+historical_steps)
        self.fill_probability = fill_probability
        
        self.with_wallet = with_wallet
        line1 =  fig.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
        line2 =  fig3.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
    
    def run_historical(self):
        
        pt = np.zeros(self.historical_steps)
        random_start_value = np.random.randint(1,99)
        
        pt[0] = random_start_value
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_historical,self.historical_steps)
        
        drift_term = np.random.normal(0,self.historical_drift_scalar,self.historical_steps)
       
        # have the figure here and have the function here
        for i in range(1,self.historical_steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
        
        self.historical_log = np.log(pt)
        self.historical_price = pt[-1]
        self.historical_mean = pt.mean()
        self.pt = pt
        
        self.lt = (1 - self.historical_log.var() * self.lamb) * (self.historical_log[-1] - self.historical_log.std())
        
        self.ut = (1 + self.historical_log.var() * self.lamb) * (self.historical_log[-1] + self.historical_log.std())
        
        self.lt = np.full(self.historical_steps+self.forward_steps,self.lt)
        self.ut = np.full(self.historical_steps+self.forward_steps,self.ut)
        
    
    
    
    def simulate_forward(self): # needs to return most recent price 
        
        steps = self.historical_steps + self.forward_steps
        self.lt = np.full(steps,self.lt)
        self.ut = np.full(steps,self.ut)
        
        pt = np.full(steps,self.historical_price)
        
        
        pt[0] = self.historical_price
        
        pt = np.concatenate([self.pt,pt],axis=0)
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_forward,steps)
        
        drift_term = np.random.normal(0,self.forward_drift_scalar,steps)
        
        
        
        display(fig)
        if self.with_wallet:
            display(fig4)
        # have the figure here and have the function here
        for i in range(self.historical_steps +1 ,steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] *  self.historical_mean/steps + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
            pt[i:steps] = np.full((steps-i),pt[i])
            
            
            if not self.regime_shift and np.log(pt[i]) > self.ut[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                fig4.data[0].y = self.wallet[self.historical_steps:]
                fig3.data[0].y = pt[:steps]
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = pt[self.historical_steps:steps]
                fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig.data[0].y = pt[:steps]
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
            
            
            
            elif not self.regime_shift and np.log(pt[i]) < self.lt[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                fig4.data[0].y = self.wallet[self.historical_steps:]
                fig3.data[0].y = pt[:steps]
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = pt[self.historical_steps:steps]
                fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig.data[0].y = pt[:steps]
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
                
                
            
            
            else:
                if self.regime_shift:
                    if np.log(pt[i]) < self.lt_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = pt[:steps]
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = pt[self.historical_steps:steps]
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = pt[:steps]
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                        
                    elif np.log(pt[i]) > self.ut_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                        
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = pt[:steps]
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = pt[self.historical_steps:steps]
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = pt[:steps]
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                    
                    else:
                      
                        self.wallet[i:] = np.full(steps-i,self.orderbook.get_wallet_development(pt[i],self.spread[i],self.wallet[i-1],self.fill_probability))
                    
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = pt[:steps]
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = pt[self.historical_steps:steps]
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = pt[:steps]
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        
                        
                        time.sleep(self.tick_time_seconds)
                    
                else:
                   
                    self.wallet[i:] = np.full(steps-i,self.orderbook.get_wallet_development(pt[i],self.spread[i],self.wallet[i-1],self.fill_probability))
                    
                    fig4.data[0].y = self.wallet[self.historical_steps:]
                    fig.data[0].y = pt[:steps]
                    fig.data[1].y = np.exp(self.lt)
                    fig.data[2].y = np.exp(self.ut)
                    time.sleep(self.tick_time_seconds)
            
    
        
        

sim = Simulator(extreme_event_probability_historical=0.0005,
              extreme_event_probability_forward=0.0005,
              lamb=1,
              historical_drift_scalar=1,
              forward_drift_scalar=2,
              seed=3,
              regime_lookBack=5,
              historical_steps=100,
              forward_steps=100,
              tick_time_seconds=0.1,
              wallet_value=100,
              fill_probability=0.3,
              with_wallet=False
              )
sim.run_historical()
sim.simulate_forward()


# Simulation for static and dynamic non-rolling

* Plots for dynamic barriers are created upon a regime shift

In [ ]:
import numpy as np # for computation of math functions 
from scipy.stats import genextreme # random variable continuous used for getting extreme events
import plotly.graph_objects as go
import time
import scipy.stats as stats




fig = go.FigureWidget() # instaciating it the figure widget 
fig.update_layout(title=dict(text="Price path with historical data with static boundary"),
                  xaxis=dict(title=dict(text="Time"))
                  ,yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))
fig2 = go.FigureWidget()
fig2.update_layout(title=dict(text="Forward price path with dynamic boundary"), 
                  xaxis=dict(title=dict(text="Time")),
                  yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))



fig3 = go.FigureWidget()
fig3.update_layout(title=dict(text="Price path with historical data with dynamic boundary"), 
                   xaxis=dict(title=dict(text="Time")),
                   yaxis=dict(title=dict(text="Cent price")),
                   font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))

fig4 = go.FigureWidget()
fig4.update_layout(title=dict(text="Wallet wealth path"), 
                   xaxis=dict(title=dict(text="Time")),
                   yaxis=dict(title=dict(text="Dollars in Wallet")),
                   font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))

scatter1 = fig.add_scatter(name='Contract price')
scatter2 = fig.add_scatter(name='Lower bound')
scatter3 = fig.add_scatter(name='Upper bound')


scatter4 = fig2.add_scatter(name='Contract price')
scatter5 = fig2.add_scatter(name='Lower bound')
scatter6 = fig2.add_scatter(name='Upper bound')

scatter7 = fig3.add_scatter(name='Contract price')
scatter8 = fig3.add_scatter(name='Lower bound')
scatter9 = fig3.add_scatter(name='Upper bound')

scatter10 = fig4.add_scatter(name='Wallet wealth')

class OrderBook:
    
        
    def get_wallet_development(self,st:int,spread:int,wallet:int,fill_probability:float) -> int: 
        
        fill_or_not = np.random.binomial(1,fill_probability,1)
        
        if fill_or_not[0]:
            bid = self.get_bid_price(st)
            ask = self.get_ask_price(st,spread)
            bid_ask_spread = ask - bid
        
            wallet += bid_ask_spread
            
            return wallet
        else:
            
            return wallet
        
    def get_bid_price(self,st) -> int:
        
        order_mean_price = st
        
        orderBook = np.full(1 ,order_mean_price) # vector filled with the amount of orders 

        "iterate over the amount of desired incoming orders"
        for i in range(100):
            order_amount = np.random.randint(1,99)
            IncomingPrices = np.round(np.random.standard_gamma(order_mean_price,order_amount),decimals=0) # new random data
            orderBook = np.append(orderBook,IncomingPrices) # adding the new random data to the order book
    
        kernel = stats.gaussian_kde(orderBook)
        x_grid = np.linspace(orderBook.min(),orderBook.max(),1000)
        kde_values = kernel(x_grid)
        price_at_max = x_grid[np.argmax(kde_values)]
            
        
        return price_at_max
        
        
        
        
        
        
    def get_ask_price(self,st:int,spread:int) -> int:
        
        order_mean_price = st + spread
        
        orderBook = np.full(1 ,order_mean_price) # vector filled with the amount of orders 


        "iterate over the amount of desired incoming orders"
        for i in range(100):
            order_amount = np.random.randint(1,99)
            IncomingPrices = np.round(np.random.standard_gamma(order_mean_price,order_amount),decimals=0) # new random data
            orderBook = np.append(orderBook,IncomingPrices) # adding the new random data to the order book
    
        kernel = stats.gaussian_kde(orderBook)
        x_grid = np.linspace(orderBook.min(),orderBook.max(),1000)
        kde_values = kernel(x_grid)
        price_at_max = x_grid[np.argmax(kde_values)]
            
        
        return price_at_max
        
    
    
class Simulator:
    def __init__(self,extreme_event_probability_historical:float,extreme_event_probability_forward:float,lamb:float,historical_drift_scalar:float,forward_drift_scalar:float,seed:int,regime_lookBack:int,historical_steps:int,forward_steps:int,tick_time_seconds:float,wallet_value:int,fill_probability:float,historical_VarAndVol_lookback:int,with_wallet:bool) -> None:
        self.extreme_event_probability_historical = float(extreme_event_probability_historical)
        self.extreme_event_probability_forward = float(extreme_event_probability_forward)
        self.wallet = np.full(forward_steps+historical_steps,wallet_value)
        
        self.lamb = lamb
        self.historical_drift_scalar = historical_drift_scalar
        self.forward_drift_scalar = forward_drift_scalar
        self.seed = seed
        self.regime_lookBack = regime_lookBack
        np.random.seed(self.seed)
        self.regime_shift = False
        self.historical_steps = historical_steps
        self.forward_steps = forward_steps
        self.tick_time_seconds = tick_time_seconds
        self.orderbook = OrderBook()
        self.fill_probability = fill_probability
        self.spread = np.random.gamma(2,2,forward_steps+historical_steps)
        self.historical_VarAndVol_lookback = historical_VarAndVol_lookback
        self.with_wallet = with_wallet
        
        line1 =  fig.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
        line2 =  fig3.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
    
    def run_historical(self):
        
        pt = np.zeros(self.historical_steps)
        random_start_value = np.random.randint(1,99)
        
        pt[0] = random_start_value
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_historical,self.historical_steps)
        
        drift_term = np.random.normal(0,self.historical_drift_scalar,self.historical_steps)
       
        # have the figure here and have the function here
        for i in range(1,self.historical_steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
        
        self.historical_log = np.log(pt[self.historical_VarAndVol_lookback:-1])
        self.historical_price = pt[-1]
        self.historical_mean = pt.mean()
        self.pt = pt
        
        self.lt = (1 - self.historical_log.var() * self.lamb) * (self.historical_log[-1] - self.historical_log.std())
        
        self.ut = (1 + self.historical_log.var() * self.lamb) * (self.historical_log[-1] + self.historical_log.std())
        
        self.lt = np.full(self.historical_steps+self.forward_steps,self.lt)
        self.ut = np.full(self.historical_steps+self.forward_steps,self.ut)
        
       
    
    
    
    
    
    
    
    
    def simulate_forward(self): # needs to return most recent price 
        
        steps = self.historical_steps + self.forward_steps
        self.lt = np.full(steps,self.lt)
        self.ut = np.full(steps,self.ut)
        
        pt = np.full(steps,self.historical_price)
        
        
        pt[0] = self.historical_price
        
        pt = np.concatenate([self.pt,pt],axis=0)
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_forward,steps)
        
        drift_term = np.random.normal(0,self.forward_drift_scalar,steps)
        
        
        
        
        display(fig)
        if self.with_wallet:
            display(fig4)
        # have the figure here and have the function here
        for i in range(self.historical_steps +1 ,steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] *  self.historical_mean/steps + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
            pt[i:steps] = np.full((steps-i),pt[i])
            
            
            if not self.regime_shift and np.log(pt[i]) > self.ut[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                fig4.data[0].y = self.wallet[self.historical_steps:]
                fig3.data[0].y = (pt[:steps])
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = (pt[self.historical_steps:steps])
                fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig.data[0].y = (pt[:steps])
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
            
            
            
            elif not self.regime_shift and np.log(pt[i]) < self.lt[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                fig4.data[0].y = self.wallet[self.historical_steps:]
                fig3.data[0].y = (pt[:steps])
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = (pt[self.historical_steps:steps])
                fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig.data[0].y = (pt[:steps])
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
                
                
            
            
            else:
                if self.regime_shift:
                    if np.log(pt[i]) < self.lt_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = (pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                        
                    elif np.log(pt[i]) > self.ut_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                        
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = np.log(pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                    
                    else:
                        
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.wallet[i:] = np.full(steps-i,self.orderbook.get_wallet_development(pt[i],self.spread[i],self.wallet[i-1],self.fill_probability))
                    
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = (pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                    
                else:
                    self.wallet[i:] = np.full(steps-i,self.orderbook.get_wallet_development(pt[i],self.spread[i],self.wallet[i-1],self.fill_probability))
                    
                    fig4.data[0].y = self.wallet[self.historical_steps:]
                    fig.data[0].y = (pt[:steps])
                    fig.data[1].y = np.exp(self.lt)
                    fig.data[2].y = np.exp(self.ut)
                    time.sleep(self.tick_time_seconds)
            

        
        

sim = Simulator(extreme_event_probability_historical=0.005,
              extreme_event_probability_forward=0.01,
              lamb=1,
              historical_drift_scalar=0.3,
              forward_drift_scalar=2,
              seed=14890,
              regime_lookBack=5,
              historical_steps=100,
              forward_steps=100,
              tick_time_seconds=0.5,
              wallet_value=100,
              fill_probability=0.3,
              historical_VarAndVol_lookback=50,
              with_wallet=False
              )
sim.run_historical()
sim.simulate_forward()


# Simulation for static and dynamic rolling

In [ ]:
import numpy as np # for computation of math functions 
from scipy.stats import genextreme # random variable continuous used for getting extreme events
import plotly.graph_objects as go
import time
import scipy.stats as stats




fig = go.FigureWidget() # instaciating it the figure widget 
fig.update_layout(title=dict(text="Price path with historical data with static boundary"),
                  xaxis=dict(title=dict(text="Time"))
                  ,yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))
fig2 = go.FigureWidget()
fig2.update_layout(title=dict(text="Forward price path with dynamic boundary"), 
                  xaxis=dict(title=dict(text="Time")),
                  yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))



fig3 = go.FigureWidget()
fig3.update_layout(title=dict(text="Price path with historical data with dynamic boundary"), 
                   xaxis=dict(title=dict(text="Time")),
                   yaxis=dict(title=dict(text="Cent price")),
                   font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))

fig4 = go.FigureWidget()
fig4.update_layout(title=dict(text="Wallet wealth path"), 
                   xaxis=dict(title=dict(text="Time")),
                   yaxis=dict(title=dict(text="Dollars in Wallet")),
                   font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))

scatter1 = fig.add_scatter(name='Contract price')
scatter2 = fig.add_scatter(name='Lower bound')
scatter3 = fig.add_scatter(name='Upper bound')


scatter4 = fig2.add_scatter(name='Contract price')
scatter5 = fig2.add_scatter(name='Lower bound')
scatter6 = fig2.add_scatter(name='Upper bound')

scatter7 = fig3.add_scatter(name='Contract price')
scatter8 = fig3.add_scatter(name='Lower bound')
scatter9 = fig3.add_scatter(name='Upper bound')

scatter10 = fig4.add_scatter(name='Wallet wealth')

class OrderBook:
    
        
    def get_wallet_development(self,st:int,spread:int,wallet:int,fill_probability:float) -> int: 
        
        fill_or_not = np.random.binomial(1,fill_probability,1)
        
        if fill_or_not[0]:
            bid = self.get_bid_price(st)
            ask = self.get_ask_price(st,spread)
            bid_ask_spread = ask - bid
        
            wallet += bid_ask_spread
            
            return wallet
        else:
            
            return wallet
        
    def get_bid_price(self,st) -> int:
        
        order_mean_price = st
        
        orderBook = np.full(1 ,order_mean_price) # vector filled with the amount of orders 

        "iterate over the amount of desired incoming orders"
        for i in range(100):
            order_amount = np.random.randint(1,99)
            IncomingPrices = np.round(np.random.standard_gamma(order_mean_price,order_amount),decimals=0) # new random data
            orderBook = np.append(orderBook,IncomingPrices) # adding the new random data to the order book
    
        kernel = stats.gaussian_kde(orderBook)
        x_grid = np.linspace(orderBook.min(),orderBook.max(),1000)
        kde_values = kernel(x_grid)
        price_at_max = x_grid[np.argmax(kde_values)]
            
        
        return price_at_max
        
        
        
        
        
        
    def get_ask_price(self,st:int,spread:int) -> int:
        
        order_mean_price = st + spread
        
        orderBook = np.full(1 ,order_mean_price) # vector filled with the amount of orders 


        "iterate over the amount of desired incoming orders"
        for i in range(100):
            order_amount = np.random.randint(1,99)
            IncomingPrices = np.round(np.random.standard_gamma(order_mean_price,order_amount),decimals=0) # new random data
            orderBook = np.append(orderBook,IncomingPrices) # adding the new random data to the order book
    
        kernel = stats.gaussian_kde(orderBook)
        x_grid = np.linspace(orderBook.min(),orderBook.max(),1000)
        kde_values = kernel(x_grid)
        price_at_max = x_grid[np.argmax(kde_values)]
            
        
        return price_at_max
        
    
    
class Simulator:
    def __init__(self,extreme_event_probability_historical:float,extreme_event_probability_forward:float,lamb:float,historical_drift_scalar:float,forward_drift_scalar:float,seed:int,regime_lookBack:int,historical_steps:int,forward_steps:int,tick_time_seconds:float,wallet_value:int,fill_probability:float,historical_VarAndVol_lookback:int,with_wallet:bool) -> None:
        self.extreme_event_probability_historical = float(extreme_event_probability_historical)
        self.extreme_event_probability_forward = float(extreme_event_probability_forward)
        self.wallet = np.full(forward_steps+historical_steps,wallet_value)
        
        self.lamb = lamb
        self.historical_drift_scalar = historical_drift_scalar
        self.forward_drift_scalar = forward_drift_scalar
        self.seed = seed
        self.regime_lookBack = regime_lookBack
        np.random.seed(self.seed)
        self.regime_shift = False
        self.historical_steps = historical_steps
        self.forward_steps = forward_steps
        self.tick_time_seconds = tick_time_seconds
        self.orderbook = OrderBook()
        self.fill_probability = fill_probability
        self.spread = np.random.gamma(2,2,forward_steps+historical_steps)
        self.historical_VarAndVol_lookback = historical_VarAndVol_lookback
        self.with_wallet = with_wallet
        
        line1 =  fig.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
        line2 =  fig3.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
    
    def run_historical(self):
        
        pt = np.zeros(self.historical_steps)
        random_start_value = np.random.randint(1,99)
        
        pt[0] = random_start_value
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_historical,self.historical_steps)
        
        drift_term = np.random.normal(0,self.historical_drift_scalar,self.historical_steps)
       
        # have the figure here and have the function here
        for i in range(1,self.historical_steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
        
        self.historical_log = np.log(pt[self.historical_VarAndVol_lookback:-1])
        self.historical_price = pt[-1]
        self.historical_mean = pt.mean()
        self.pt = pt
        
        self.lt = (1 - self.historical_log.var() * self.lamb) * (self.historical_log[-1] - self.historical_log.std())
        
        self.ut = (1 + self.historical_log.var() * self.lamb) * (self.historical_log[-1] + self.historical_log.std())
        
        self.lt = np.full(self.historical_steps+self.forward_steps,self.lt)
        self.ut = np.full(self.historical_steps+self.forward_steps,self.ut)
        
       
    
    
    
    
    
    
    
    
    def simulate_forward(self): # needs to return most recent price 
        
        steps = self.historical_steps + self.forward_steps
        self.lt = np.full(steps,self.lt)
        self.ut = np.full(steps,self.ut)
        
        pt = np.full(steps,self.historical_price)
        
        
        pt[0] = self.historical_price
        
        pt = np.concatenate([self.pt,pt],axis=0)
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_forward,steps)
        
        drift_term = np.random.normal(0,self.forward_drift_scalar,steps)
        
        
        
        
        display(fig)
        if self.with_wallet:
            display(fig4)
        # have the figure here and have the function here
        for i in range(self.historical_steps +1 ,steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] *  self.historical_mean/steps + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
            pt[i:steps] = np.full((steps-i),pt[i])
            
            
            if not self.regime_shift and np.log(pt[i]) > self.ut[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                fig4.data[0].y = self.wallet[self.historical_steps:]
                fig3.data[0].y = (pt[:steps])
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = (pt[self.historical_steps:steps])
                fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig.data[0].y = (pt[:steps])
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
            
            
            
            elif not self.regime_shift and np.log(pt[i]) < self.lt[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                fig4.data[0].y = self.wallet[self.historical_steps:]
                fig3.data[0].y = (pt[:steps])
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = (pt[self.historical_steps:steps])
                fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig.data[0].y = (pt[:steps])
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
                
                
            
            
            else:
                if self.regime_shift:
                    if np.log(pt[i]) < self.lt_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = (pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                        
                    elif np.log(pt[i]) > self.ut_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                        
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.wallet[i:] = np.full(steps-i,self.wallet[i-1])
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = np.log(pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                    
                    else:
                        
                    
                        fig4.data[0].y = self.wallet[self.historical_steps:]
                        fig3.data[0].y = (pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[1].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                    
                else:
                    self.wallet[i:] = np.full(steps-i,self.orderbook.get_wallet_development(pt[i],self.spread[i],self.wallet[i-1],self.fill_probability))
                    
                    fig4.data[0].y = self.wallet[self.historical_steps:]
                    fig.data[0].y = (pt[:steps])
                    fig.data[1].y = np.exp(self.lt)
                    fig.data[2].y = np.exp(self.ut)
                    time.sleep(self.tick_time_seconds)
            

        
        

sim = Simulator(extreme_event_probability_historical=0.005,
              extreme_event_probability_forward=0.01,
              lamb=1,
              historical_drift_scalar=0.3,
              forward_drift_scalar=2,
              seed=14890,
              regime_lookBack=5,
              historical_steps=100,
              forward_steps=100,
              tick_time_seconds=1,
              wallet_value=100,
              fill_probability=0.3,
              historical_VarAndVol_lookback=5,
              with_wallet=False
              )
sim.run_historical()
sim.simulate_forward()


# With midqoute fair price

This is a version not detailed in the paper where it uses the mid quote of the boundaries to find the optimal quote price

In [ ]:
import numpy as np # for computation of math functions 
from scipy.stats import genextreme # random variable continuous used for getting extreme events

import plotly.graph_objects as go
import time






fig = go.FigureWidget() # instaciating it the figure widget 
fig.update_layout(title=dict(text="Price path with historical data, and with non moving boundary"),
                  xaxis=dict(title=dict(text="Time"))
                  ,yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))


fig2 = go.FigureWidget()
fig2.update_layout(title=dict(text="Forward price path with moving boundary"), 
                  xaxis=dict(title=dict(text="Time")),
                  yaxis=dict(title=dict(text="Cent price")),
                  font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"))



fig3 = go.FigureWidget()
fig3.update_layout(title=dict(text="Price path with historical data, and with moving boundary"), 
                   xaxis=dict(title=dict(text="Time")),
                   yaxis=dict(title=dict(text="Cent price")),
                   font=dict(family="Courier New, monospace",size=14,color="RebeccaPurple"),
                   )



scatter1 = fig.add_scatter(name='Contract price')
scatter2 = fig.add_scatter(name='Lower bound')
scatter3 = fig.add_scatter(name='Upper bound')


scatter4 = fig2.add_scatter(name='Contract price')
scatter5 = fig2.add_scatter(name='Fair Price',line=dict(color='deepskyblue', width=2))
scatter6 = fig2.add_scatter(name='Upper bound')
scatter7 = fig2.add_scatter(name='Lower bound',line=dict(color='orangered', width=2))

scatter8 = fig3.add_scatter(name='Contract price')
scatter9 = fig3.add_scatter(name='Lower bound')
scatter10 = fig3.add_scatter(name='Upper bound')




        
    
    
class Simulator:
    def __init__(self,extreme_event_probability_historical:float,extreme_event_probability_forward:float,lamb:float,historical_drift_scalar:float,forward_drift_scalar:float,seed:int,regime_lookBack:int,historical_steps:int,forward_steps:int,tick_time_seconds:float,fill_probability:float) -> None:
        self.extreme_event_probability_historical = float(extreme_event_probability_historical)
        self.extreme_event_probability_forward = float(extreme_event_probability_forward)
        
        self.lamb = lamb
        self.historical_drift_scalar = historical_drift_scalar
        self.forward_drift_scalar = forward_drift_scalar
        self.seed = seed
        self.regime_lookBack = regime_lookBack
        np.random.seed(self.seed)
        self.regime_shift = False
        self.historical_steps = historical_steps
        self.forward_steps = forward_steps
        self.tick_time_seconds = tick_time_seconds
        self.fill_probability = fill_probability
        self.spread = np.random.gamma(2,2,forward_steps+historical_steps)
        
        line1 =  fig.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
        line2 =  fig3.add_vline(x=historical_steps, line_width=3, line_dash="dash", line_color="green",name='Forward simulation start')
    
    def run_historical(self):
        
        pt = np.zeros(self.historical_steps)
        random_start_value = np.random.randint(1,99)
        
        pt[0] = random_start_value
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_historical,self.historical_steps)
        
        drift_term = np.random.normal(0,self.historical_drift_scalar,self.historical_steps)
       
        # have the figure here and have the function here
        for i in range(1,self.historical_steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
        
        self.historical_log = np.log(pt)
        self.historical_price = pt[-1]
        self.historical_mean = pt.mean()
        self.pt = pt
        
        self.lt = (1 - self.historical_log.var() * self.lamb) * (self.historical_log[-1] - self.historical_log.std())
        
        self.ut = (1 + self.historical_log.var() * self.lamb) * (self.historical_log[-1] + self.historical_log.std())
        
        self.lt = np.full(self.historical_steps+self.forward_steps,self.lt)
        self.ut = np.full(self.historical_steps+self.forward_steps,self.ut)
        
       
    
    
    
    
    
    
    
    
    def simulate_forward(self): # needs to return most recent price 
        
        steps = self.historical_steps + self.forward_steps
        self.lt = np.full(steps,self.lt)
        self.ut = np.full(steps,self.ut)
        self.fair_price = np.full(steps,(self.lt[-1] + self.ut[-1])/2)
        
        pt = np.full(steps,self.historical_price)
        
        
        pt[0] = self.historical_price
        
        pt = np.concatenate([self.pt,pt],axis=0)
        
        extreme_event = np.random.binomial(1,self.extreme_event_probability_forward,steps)
        
        drift_term = np.random.normal(0,self.forward_drift_scalar,steps)
        
        
        
        display(fig)
        # have the figure here and have the function here
        for i in range(self.historical_steps +1 ,steps):
            if extreme_event[i]:
                extreme_event[i] = genextreme.rvs(1,30,50)
                
            
            pt[i] = pt[i-1] + drift_term[i] *  self.historical_mean/steps + extreme_event[i]  #type: ignore
            
            if pt[i] >= 99:
                pt[i] = 99
                
            if pt[i] <= 1:
                pt[i] = 1
            
            pt[i:steps] = np.full((steps-i),pt[i])
            
            
            if not self.regime_shift and np.log(pt[i]) > self.ut[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.fair_price[i:steps] = np.full((steps-i),(self.lt_regime[i] + self.ut_regime[i])/2)

                        
                fig3.data[0].y = (pt[:steps])
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = (pt[self.historical_steps:steps])
                fig2.data[3].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig2.data[1].y = np.exp(self.fair_price[self.historical_steps:])
                fig.data[0].y = (pt[:steps])
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
            
            
            
            elif not self.regime_shift and np.log(pt[i]) < self.lt[i]:
                log_price = np.log(pt[i-self.regime_lookBack:i])
                lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                self.regime_shift = True
                display(fig3)
                display(fig2)
                    
                self.lt_regime  = np.full(steps,lt)
                self.ut_regime  = np.full(steps,ut)
                self.fair_price[i:steps] = np.full((steps-i),(self.lt_regime[i] + self.ut_regime[i])/2)

                        
                fig3.data[0].y = (pt[:steps])
                fig3.data[1].y = np.exp(self.lt_regime)
                fig3.data[2].y = np.exp(self.ut_regime)
                fig2.data[0].y = (pt[self.historical_steps:steps])
                fig2.data[3].y = np.exp(self.lt_regime[self.historical_steps:])
                fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                fig2.data[1].y = np.exp(self.fair_price[self.historical_steps:])
                fig.data[0].y = (pt[:steps])
                fig.data[1].y = np.exp(self.lt)
                fig.data[2].y = np.exp(self.ut)
                time.sleep(self.tick_time_seconds)
                
                
            
            
            else:
                if self.regime_shift:
                    if np.log(pt[i]) < self.lt_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.fair_price[i:steps] = np.full((steps-i),(self.lt_regime[i] + self.ut_regime[i])/2)

                        
                        fig3.data[0].y = (pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[3].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig2.data[1].y = np.exp(self.fair_price[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                        
                    elif np.log(pt[i]) > self.ut_regime[i]:
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        self.regime_shift = True
                        
                    
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.fair_price[i:steps] = np.full((steps-i),(self.lt_regime[i] + self.ut_regime[i])/2)

                        
                        fig3.data[0].y = (pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[3].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig2.data[1].y = np.exp(self.fair_price[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                    
                    else:
                        
                        log_price = np.log(pt[i-self.regime_lookBack:i])
                        lt = (1 - log_price.var() * self.lamb) * (log_price[-1] - log_price.std())
                        ut = (1 + log_price.var() * self.lamb) * (log_price[-1] + log_price.std())
                        
                        self.lt_regime  = np.full(steps,lt)
                        self.ut_regime  = np.full(steps,ut)
                        self.fair_price[i:steps] = np.full((steps-i),(self.lt_regime[i] + self.ut_regime[i])/2)

                        
                        fig3.data[0].y = (pt[:steps])
                        fig3.data[1].y = np.exp(self.lt_regime)
                        fig3.data[2].y = np.exp(self.ut_regime)
                        fig2.data[0].y = (pt[self.historical_steps:steps])
                        fig2.data[3].y = np.exp(self.lt_regime[self.historical_steps:])
                        fig2.data[2].y = np.exp(self.ut_regime[self.historical_steps:])
                        fig2.data[1].y = np.exp(self.fair_price[self.historical_steps:])
                        fig.data[0].y = (pt[:steps])
                        fig.data[1].y = np.exp(self.lt)
                        fig.data[2].y = np.exp(self.ut)
                        time.sleep(self.tick_time_seconds)
                    
                else:
                    
                    fig.data[0].y = (pt[:steps])
                    fig.data[1].y = np.exp(self.lt)
                    fig.data[2].y = np.exp(self.ut)
                    time.sleep(self.tick_time_seconds)
                
        
        

sim = Simulator(extreme_event_probability_historical=0.005,
              extreme_event_probability_forward=0.005,
              lamb=1,
              historical_drift_scalar=0.3,
              forward_drift_scalar=2,
              seed=7223,
              regime_lookBack=5,
              historical_steps=100,
              forward_steps=100,
              tick_time_seconds=0.07,
              fill_probability=0.3)
sim.run_historical()
sim.simulate_forward()
